# CSV-Export: Mapillary Coverage für OSM Highways

Dieses Skript kombiniert die Coverage-Daten (Pano + Regular) aus Schritt 3 und erstellt:
- Eine finale CSV-Datei mit allen OSM-Highway-Segmenten, die >= 60% Mapillary-Coverage haben
- Eine README.md mit Zusammenfassung nach Bundesland

In [1]:
import pandas as pd
import geopandas as gpd

from datetime import datetime
from pathlib import Path
import json
import glob
import os

from config import TILES_CONFIG, PROCESSING_CONFIG, MAPILLARY_CONFIG, GEOFABRIK_CONFIG

# Konfiguration
OUTPUT_FOLDER = PROCESSING_CONFIG["output_folder"]
COVERAGE_THRESHOLD = PROCESSING_CONFIG['mp_coverage_ratio_threshold']

In [2]:
def should_process_csv(output_file, max_age_days=None):
    """
    Prüft ob CSV neu erstellt werden muss.
    
    Args:
        output_file: Pfad zur CSV-Datei
        max_age_days: Maximales Alter in Tagen (default: aus config)
    
    Returns:
        True wenn CSV-Erstellung nötig, False sonst
    """
    if max_age_days is None:
        max_age_days = PROCESSING_CONFIG.get('max_file_age_days', 4)
    
    if not os.path.exists(output_file):
        return True
    
    try:
        file_age_days = (datetime.now().timestamp() - os.path.getmtime(output_file)) / 86400
        
        if file_age_days >= max_age_days:
            print(f"  ℹ️  CSV ist {file_age_days:.1f} Tage alt → Erneuerung nötig")
            return True
        else:
            print(f"  ✓ CSV ist {file_age_days:.1f} Tage alt → aktuell genug")
            return False
            
    except Exception as e:
        print(f"  ⚠️  Fehler beim Prüfen: {e}")
        return True

In [3]:
# Prüfe ob CSV-Export nötig ist
csv_output_file = f"{OUTPUT_FOLDER}/germany_osm-highways_mp-coverage_latest.csv"

print(f"{'='*70}")
print(f"CSV-Export Prüfung")
print(f"{'='*70}")

if not should_process_csv(csv_output_file):
    print(f"\n✅ CSV ist aktuell genug - keine Verarbeitung nötig!")
    print(f"   Datei: {csv_output_file}")
    print(f"   Wenn du trotzdem neu erstellen möchtest, lösche die Datei.")
    SHOULD_PROCESS = False
else:
    print(f"\n▶️  Fahre fort mit CSV-Erstellung...")
    SHOULD_PROCESS = True

print(f"{'='*70}")

CSV-Export Prüfung
  ℹ️  CSV ist 7.0 Tage alt → Erneuerung nötig

▶️  Fahre fort mit CSV-Erstellung...


In [4]:
#SHOULD_PROCESS = True


In [5]:
def load_metadata():
    """Lädt OSM und Mapillary Metadaten aus JSON-Dateien."""
    files = {
        "output/osm_metadata.json": "osm_data_from",
        "output/ml_metadata.json": "ml_data_from",
    }
    
    metadata = {
        "osm_data_from": None, 
        "ml_data_from": None,
        "osm_bundeslaender": {},
        "ml_bundeslaender": {}
    }
    
    for fname, primary_key in files.items():
        p = Path(fname)
        if not p.exists():
            print(f"⚠️  {p} nicht gefunden")
            continue
        
        with p.open("r", encoding="utf-8") as f:
            meta = json.load(f)
        
        if primary_key in meta:
            metadata[primary_key] = meta[primary_key]
        
        # Lade auch Bundesland-spezifische Timestamps
        if "bundeslaender" in meta:
            if "osm" in fname:
                metadata["osm_bundeslaender"] = meta["bundeslaender"]
            elif "ml" in fname:
                metadata["ml_bundeslaender"] = meta["bundeslaender"]
    
    return metadata

# Lade Metadaten
metadata = load_metadata()
print(f"OSM Daten von: {metadata['osm_data_from']}")
print(f"Mapillary Daten von: {metadata['ml_data_from']}")
print(f"OSM Bundesländer: {len(metadata['osm_bundeslaender'])}")
print(f"ML Bundesländer: {len(metadata['ml_bundeslaender'])}")

OSM Daten von: 2026-03-08T21:20:46Z
Mapillary Daten von: 2026-03-09T01:54:50.435944+00:00
OSM Bundesländer: 16
ML Bundesländer: 16


In [6]:
def load_coverage_data(coverage_type="pano"):
    """
    Lädt Coverage-Daten für alle Bundesländer.
    
    Args:
        coverage_type: "pano" oder "regular"
    
    Returns:
        tuple: (combined_df, summary_list)
    """
    # Finde Dateien
    files = glob.glob(f"{OUTPUT_FOLDER}/DE-*_osm-highways_mp_{coverage_type}_coverage_latest.parquet")
    
    # Filter nach ausgewählten Bundesländern
    selected_bundeslaender = GEOFABRIK_CONFIG.get("bundeslaender")
    if selected_bundeslaender:
        files = [f for f in files if any(bl in f for bl in selected_bundeslaender)]
    
    print(f"\n{'─'*70}")
    print(f"Lade {coverage_type.upper()} Coverage-Daten:")
    print(f"  Dateien gefunden: {len(files)}")
    
    all_dfs = []
    summary_list = []
    
    for file in files:
        # Lade nur benötigte Spalten
        df = pd.read_parquet(file, columns=["osm_id", "mp_coverage_ratio", "length_m_before_clip"])
        
        # Extrahiere Bundesland
        bundesland = file.split('/')[-1].split('_')[0]
        
        # Summary-Statistik
        total_length = df['length_m_before_clip'].sum()
        summary_list.append({
            'Bundesland': bundesland,
            'Typ': coverage_type,
            'Anzahl Segmente': len(df),
            'Gesamtlänge (km)': total_length / 1000
        })
        
        all_dfs.append(df)
        print(f"    {bundesland}: {len(df):,} Segmente, {total_length/1000:,.2f} km")
    
    # Kombiniere alle Bundesländer
    if all_dfs:
        combined_df = pd.concat(all_dfs, ignore_index=True)
        print(f"  ✓ Gesamt: {len(combined_df):,} Segmente")
        return combined_df, summary_list
    else:
        print(f"  ⚠️  Keine Daten gefunden!")
        return pd.DataFrame(columns=["osm_id", "mp_coverage_ratio"]), []

# Lade Daten nur wenn nötig
if SHOULD_PROCESS:
    gdf_pano, pano_summary = load_coverage_data("pano")
    gdf_regular, regular_summary = load_coverage_data("regular")
else:
    gdf_pano, pano_summary = pd.DataFrame(), []
    gdf_regular, regular_summary = pd.DataFrame(), []


──────────────────────────────────────────────────────────────────────
Lade PANO Coverage-Daten:
  Dateien gefunden: 15
    DE-BW: 216,883 Segmente, 30,445.85 km
    DE-TH: 6,670 Segmente, 1,625.69 km


    DE-BB: 129,639 Segmente, 26,072.30 km
    DE-MV: 7,271 Segmente, 2,833.17 km
    DE-NW: 25,942 Segmente, 3,064.11 km


    DE-SH: 7,903 Segmente, 1,654.13 km
    DE-RP: 30,771 Segmente, 7,980.74 km


    DE-BY: 94,850 Segmente, 18,837.07 km
    DE-ST: 47,176 Segmente, 8,818.72 km
    DE-SN: 37,030 Segmente, 3,848.83 km
    DE-HH: 3,603 Segmente, 323.22 km
    DE-NI: 43,344 Segmente, 11,638.65 km
    DE-BE: 198,548 Segmente, 13,472.00 km
    DE-HB: 855 Segmente, 114.59 km
    DE-HE: 48,810 Segmente, 8,083.43 km
  ✓ Gesamt: 899,295 Segmente

──────────────────────────────────────────────────────────────────────
Lade REGULAR Coverage-Daten:
  Dateien gefunden: 16


    DE-NI: 329,140 Segmente, 70,973.86 km

...

    DE-SH: 102,236 Segmente, 22,817.66 km
    DE-HE: 212,129 Segmente, 34,419.79 km
    DE-MV: 44,303 Segmente, 11,012.66 km
    DE-NW: 609,130 Segmente, 93,107.59 km
    DE-BE: 186,590 Segmente, 12,422.07 km


    DE-TH: 72,009 Segmente, 14,309.09 km
    DE-SL: 29,241 Segmente, 5,203.71 km


    DE-BW: 484,289 Segmente, 75,775.15 km
    DE-HB: 10,651 Segmente, 1,318.28 km
    DE-BY: 431,914 Segmente, 79,430.15 km
    DE-SN: 187,736 Segmente, 27,856.83 km
    DE-BB: 178,410 Segmente, 45,395.53 km


    DE-RP: 207,521 Segmente, 41,198.90 km
    DE-ST: 94,523 Segmente, 20,276.65 km


  ✓ Gesamt: 3,261,499 Segmente


## Zusammenfassung nach Bundesland

In [7]:
if SHOULD_PROCESS and (pano_summary or regular_summary):
    # Kombiniere Summaries
    summary_df = pd.concat([
        pd.DataFrame(pano_summary),
        pd.DataFrame(regular_summary)
    ], ignore_index=True).sort_values(['Bundesland', 'Typ'])
    
    print(f"\n{'='*70}")
    print("📊 Zusammenfassung nach Bundesland und Typ:")
    print(f"{'='*70}")
    print(summary_df.to_string(index=False))
    
    # Gesamtsummen
    total_pano_km = summary_df[summary_df['Typ']=='pano']['Gesamtlänge (km)'].sum()
    total_regular_km = summary_df[summary_df['Typ']=='regular']['Gesamtlänge (km)'].sum()
    total_pano_count = summary_df[summary_df['Typ']=='pano']['Anzahl Segmente'].sum()
    total_regular_count = summary_df[summary_df['Typ']=='regular']['Anzahl Segmente'].sum()
    
    print(f"\n{'='*70}")
    print(f"Gesamt Pano:    {total_pano_km:,.2f} km ({total_pano_count:,} Segmente)")
    print(f"Gesamt Regular: {total_regular_km:,.2f} km ({total_regular_count:,} Segmente)")
    print(f"{'='*70}")
    
    display(summary_df)
else:
    summary_df = pd.DataFrame()
    print("⏩ Keine Summary zu erstellen")


📊 Zusammenfassung nach Bundesland und Typ:
Bundesland     Typ  Anzahl Segmente  Gesamtlänge (km)
     DE-BB    pano           129639      26072.299407
     DE-BB regular           178410      45395.534052
     DE-BE    pano           198548      13472.000556
     DE-BE regular           186590      12422.069040
     DE-BW    pano           216883      30445.847196
     DE-BW regular           484289      75775.149342
     DE-BY    pano            94850      18837.065919
     DE-BY regular           431914      79430.148664
     DE-HB    pano              855        114.588605
     DE-HB regular            10651       1318.284257
     DE-HE    pano            48810       8083.433523
     DE-HE regular           212129      34419.787972
     DE-HH    pano             3603        323.216448
     DE-HH regular            81677       6821.006156
     DE-MV    pano             7271       2833.171784
     DE-MV regular            44303      11012.663007
     DE-NI    pano            43344   

,Bundesland,Typ,Anzahl Segmente,Gesamtlänge (km)
2,DE-BB,pano,129639,26072.299407
28,DE-BB,regular,178410,45395.534052
12,DE-BE,pano,198548,13472.000556
21,DE-BE,regular,186590,12422.069040
0,DE-BW,pano,216883,30445.847196
24,DE-BW,regular,484289,75775.149342
7,DE-BY,pano,94850,18837.065919
26,DE-BY,regular,431914,79430.148664
13,DE-HB,pano,855,114.588605
25,DE-HB,regular,10651,1318.284257


## Filtern nach Coverage-Threshold und Kombinieren

In [8]:
def filter_and_tag(df, coverage_type):
    """
    Filtert DataFrame nach Coverage-Threshold und fügt Tag hinzu.
    
    Args:
        df: DataFrame mit osm_id und mp_coverage_ratio
        coverage_type: "pano" oder "regular"
    
    Returns:
        DataFrame mit osm_id und mapillary_coverage
    """
    if df.empty:
        return pd.DataFrame(columns=["osm_id", "mapillary_coverage"])
    
    # Behalte nur benötigte Spalten
    filtered = df[["osm_id", "mp_coverage_ratio"]].copy()
    
    # Filter nach Threshold
    filtered = filtered[filtered["mp_coverage_ratio"] >= COVERAGE_THRESHOLD].copy()
    
    # Füge Coverage-Type hinzu
    filtered["mapillary_coverage"] = coverage_type
    
    # Entferne mp_coverage_ratio
    filtered = filtered[["osm_id", "mapillary_coverage"]]
    
    print(f"  {coverage_type}: {len(filtered):,} Segmente über Threshold ({COVERAGE_THRESHOLD*100:.0f}%)")
    
    return filtered

if SHOULD_PROCESS:
    print(f"\n{'─'*70}")
    print(f"Filtere nach Coverage-Threshold >= {COVERAGE_THRESHOLD*100:.0f}%:")
    
    pano_filtered = filter_and_tag(gdf_pano, "pano")
    regular_filtered = filter_and_tag(gdf_regular, "regular")
else:
    pano_filtered = pd.DataFrame()
    regular_filtered = pd.DataFrame()


──────────────────────────────────────────────────────────────────────
Filtere nach Coverage-Threshold >= 60%:


  pano: 506,657 Segmente über Threshold (60%)


  regular: 1,805,469 Segmente über Threshold (60%)

In [9]:
if SHOULD_PROCESS:
    # Kombiniere Pano + Regular
    both_concat = pd.concat([pano_filtered, regular_filtered], ignore_index=True)
    
    print(f"\n{'─'*70}")
    print(f"Vor Duplikat-Entfernung:")
    print(both_concat["mapillary_coverage"].value_counts().to_string())
    
    # Entferne Duplikate: Pano hat Vorrang über Regular
    # (ascending=True bei sort → 'pano' kommt vor 'regular' alphabetisch)
    both_concat = both_concat.sort_values(
        by="mapillary_coverage", 
        ascending=True
    ).drop_duplicates(
        subset="osm_id", 
        keep="first"
    )
    
    print(f"\nNach Duplikat-Entfernung (Pano bevorzugt):")
    print(both_concat["mapillary_coverage"].value_counts().to_string())
    print(f"\nGesamt eindeutige Segmente: {len(both_concat):,}")
    
    display(both_concat.head(10))
else:
    both_concat = pd.DataFrame()
    print("⏩ Keine Kombination nötig")


──────────────────────────────────────────────────────────────────────
Vor Duplikat-Entfernung:


mapillary_coverage
regular    1805469
pano        506657



Nach Duplikat-Entfernung (Pano bevorzugt):
mapillary_coverage
regular    1582401
pano        506592

Gesamt eindeutige Segmente: 2,088,993


,osm_id,mapillary_coverage
0,1451723323,pano
337778,984701054,pano
337777,984700129,pano
337776,984700128,pano
337775,984700127,pano
337774,984609021,pano
337773,984609020,pano
337772,984046073,pano
337771,984046072,pano
337770,984042141,pano


In [10]:
## CSV und README exportieren

In [11]:
if SHOULD_PROCESS and not both_concat.empty:
    # Speichere CSV
    both_concat.to_csv(csv_output_file, index=False)
    
    print(f"\n{'='*70}")
    print(f"✅ CSV exportiert:")
    print(f"   {csv_output_file}")
    print(f"   {len(both_concat):,} Zeilen")
    print(f"{'='*70}")
else:
    print("\n⏩ Kein CSV-Export")


✅ CSV exportiert:
   output/germany_osm-highways_mp-coverage_latest.csv
   2,088,993 Zeilen


In [12]:
def create_readme(summary_df, metadata):
    """Erstellt README.md mit Zusammenfassung."""
    current_date = datetime.now().strftime("%Y-%m-%d")
    
    # Pivot-Tabelle für Markdown
    summary_table = summary_df.pivot(
        index='Bundesland', 
        columns='Typ', 
        values='Gesamtlänge (km)'
    ).fillna(0).reset_index()
    
    # Hole Bundesland-spezifische Timestamps
    osm_bl = metadata.get("osm_bundeslaender", {})
    ml_bl = metadata.get("ml_bundeslaender", {})
    
    # Markdown-Tabelle erstellen mit zusätzlichen Datums-Spalten
    table_header = "| Bundesland | Pano (km) | Regular (km) | OSM Datum | Mapillary Datum |\n|------------|-----------|--------------|-----------|-----------------|"
    table_rows = []
    
    for _, row in summary_table.iterrows():
        bundesland = row['Bundesland']
        pano_km = f"{row.get('pano', 0):,.2f}" if 'pano' in row else "0.00"
        regular_km = f"{row.get('regular', 0):,.2f}" if 'regular' in row else "0.00"
        
        # Extrahiere Datum (nur YYYY-MM-DD)
        osm_date = osm_bl.get(bundesland, "N/A")
        ml_date = ml_bl.get(bundesland, "N/A")
        
        if osm_date != "N/A":
            osm_date = osm_date.split('T')[0]
        if ml_date != "N/A":
            ml_date = ml_date.split('T')[0]
        
        table_rows.append(f"| {bundesland} | {pano_km} | {regular_km} | {osm_date} | {ml_date} |")
    
    markdown_table = "\n".join([table_header] + table_rows)
    
    # README-Inhalt
    osm_date = metadata['osm_data_from'].split('T')[0] if metadata['osm_data_from'] else "N/A"
    ml_date = metadata['ml_data_from'].split('T')[0] if metadata['ml_data_from'] else "N/A"
    
    readme_content = f"""# Mapillary Coverage per OSM Highway — Output

This folder contains the **latest** output file for *Mapillary coverage per OSM highway analysis*.

| Property | Value |
|----------|-------|
| **Data created** | {current_date} |
| **OSM data** | {osm_date} |
| **Mapillary data** | {PROCESSING_CONFIG['min_capture_date']} → {ml_date} |
| **Buffer distance** | {PROCESSING_CONFIG['buffer_distance']} meters |
| **Coverage ratio threshold** | {PROCESSING_CONFIG['mp_coverage_ratio_threshold']} ({int(PROCESSING_CONFIG['mp_coverage_ratio_threshold']*100)}%) |

Segments are considered *covered* if at least {int(PROCESSING_CONFIG['mp_coverage_ratio_threshold']*100)}% of their length falls within {PROCESSING_CONFIG['buffer_distance']} meters of Mapillary images.

## Summary by Bundesland

{markdown_table}
"""
    
    return readme_content

if SHOULD_PROCESS and not summary_df.empty:
    readme_content = create_readme(summary_df, metadata)
    
    readme_path = f"{OUTPUT_FOLDER}/README.md"
    with open(readme_path, "w", encoding="utf-8") as f:
        f.write(readme_content)
    
    print(f"\n✅ README erstellt: {readme_path}")
else:
    print("\n⏩ Kein README-Export")

print(f"\n{'='*70}")
print("✅ FERTIG!")
print(f"{'='*70}")


✅ README erstellt: output/README.md

✅ FERTIG!
